# Module 2 · Lesson 05: Prompt Evaluation

How do you know if your prompt is **good**? In production, we use **LLM-as-Judge**
to automatically evaluate prompt quality.

## What you will learn
1. Why prompt evaluation matters
2. **LLM-as-Judge** — using one model to evaluate another
3. Building a **scoring rubric**
4. **A/B testing** prompts
5. Batch evaluation for consistency

In [1]:
# ── Setup ──────────────────────────────────────────────
import os, json
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI
client = OpenAI()

def ask(prompt, system=None, temperature=0.7, max_tokens=400):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

if client:
    print("Ready")

Ready


---
## 1. LLM-as-Judge Pattern

Use a **stronger model** (or the same model with a judge prompt) to score outputs.

```
Prompt A → Model → Response A ─┐
                                ├─→ Judge (LLM) → Score
Rubric ─────────────────────────┘
```

In [2]:
def evaluate_response(question: str, response: str, criteria: str) -> dict:
    """Have Claude act as an LLM judge and return structured JSON."""
    eval_system = (
        "You are an expert prompt-engineering evaluator.\n"
        "You will receive a QUESTION, a RESPONSE, and CRITERIA.\n\n"
        "Score EACH dimension 1-10, then compute the average as 'score'.\n\n"
        "Return ONLY valid JSON (no markdown fences). Schema:\n"
        '{"score": <float>, "breakdown": {"clarity": <int>, "analogy": <int>, '
        '"conciseness": <int>, "actionable": <int>}, "one_liner": "<= 15 word summary"}'
    )

    eval_prompt = (
        f"QUESTION:\n{question}\n\n"
        f"RESPONSE:\n{response}\n\n"
        f"CRITERIA:\n{criteria}"
    )

    raw = ask(eval_prompt, system=eval_system, temperature=0.0)

    try:
        cleaned = raw.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned.removeprefix("```json").removesuffix("```").strip()
        elif cleaned.startswith("```"):
            cleaned = cleaned.removeprefix("```").removesuffix("```").strip()

        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"score": 0, "breakdown": {}, "one_liner": "⚠ JSON parse error"}

---
## 2. A/B Testing Prompts

Compare two different prompts on the **same question**:

In [3]:
PCTF_PROMPT = """
You are a senior DevOps engineer mentoring a junior developer on their first day.

Purpose:
Answer the question: "What is Docker?"

Context:
The junior knows basic programming but is new to DevOps, deployment, and containers.

Task:
Write a crisp, vivid explanation of Docker in exactly 3 sentences, between 45 and 55 words total.

Evaluation targets:
- Clarity: every sentence should feel natural when read aloud — plain language a beginner grasps instantly
- Analogy: weave exactly one shipping-container analogy that highlights portability — the reader should visualize why standardization matters
- Conciseness: active voice, strong verbs, zero filler — if a word can be removed without losing meaning, remove it
- Actionable: end with a terminal command and what the junior will see when they run it

Output constraints:
- No sentence may exceed 20 words
- Do not use jargon that needs explanation
- Do not use filler phrases like "allows developers to", "to get started", "you can check if"
- Do not use: "lightweight", "seamlessly", "efficiently", "similar to how"
- No bullet points, headings, or code fences
"""


In [4]:
question = "Explain what Docker is to a junior developer."

# Prompt A: Simple
response_a = ask(question)

# Prompt B: PCTF
response_b = ask(
    question,
    system=PCTF_PROMPT,
)

criteria = (
    "Clarity for beginners (0-10), "
    "quality of analogies (0-10), "
    "conciseness — shorter is better (0-10), "
    "actionable next-step the reader can try immediately (0-10). "
    "Score each dimension then average for the final score."
)

eval_a = evaluate_response(question, response_a, criteria)
eval_b = evaluate_response(question, response_b, criteria)

words_a = len(response_a.split())
words_b = len(response_b.split())

winner = "A" if eval_a.get("score", 0) > eval_b.get("score", 0) else "B"

md = f"""
# A/B Test Results

## Question
**{question}**

| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | {eval_a.get('score', 'N/A')}/10 | {eval_b.get('score', 'N/A')}/10 |
| **Clarity** | {eval_a.get('breakdown', {}).get('clarity', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('clarity', 'N/A')}/10 |
| **Analogy** | {eval_a.get('breakdown', {}).get('analogy', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('analogy', 'N/A')}/10 |
| **Conciseness** | {eval_a.get('breakdown', {}).get('conciseness', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('conciseness', 'N/A')}/10 |
| **Actionable** | {eval_a.get('breakdown', {}).get('actionable', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('actionable', 'N/A')}/10 |
| **Word Count** | {words_a} | {words_b} |

## Prompt A Response
{response_a[:500]}{'...' if len(response_a) > 500 else ''}

**Judge summary:** {eval_a.get('one_liner', 'N/A')}

---

## Prompt B Response
{response_b[:500]}{'...' if len(response_b) > 500 else ''}

**Judge summary:** {eval_b.get('one_liner', 'N/A')}

---

# 🏆 Winner: Prompt {winner}
"""

display(Markdown(md))


# A/B Test Results

## Question
**Explain what Docker is to a junior developer.**

| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | 7.25/10 | 8.25/10 |
| **Clarity** | 8/10 | 9/10 |
| **Analogy** | 7/10 | 8/10 |
| **Conciseness** | 6/10 | 7/10 |
| **Actionable** | 7/10 | 9/10 |
| **Word Count** | 309 | 46 |

## Prompt A Response
Sure! Docker is a platform that allows developers to automate the deployment of applications inside lightweight, portable containers. Here’s a breakdown of the key concepts:

### 1. **Containers**:
- **What is a Container?**: A container is like a lightweight, standalone package that includes everything needed to run a piece of software, such as the code, runtime, libraries, and dependencies. Think of it like a small, isolated environment that runs your application.
- **Isolation**: Each contain...

**Judge summary:** Docker simplifies application deployment using lightweight, portable containers.

---

## Prompt B Response
Docker is a tool that packages applications into containers, like shipping containers that hold goods. This makes it easy to move software between different environments without worrying about compatibility issues. Run the command "docker --version" in your terminal, and you'll see the installed Docker version displayed.

**Judge summary:** Docker packages applications into containers for easy deployment across environments.

---

# 🏆 Winner: Prompt B


---
## 3. Batch Evaluation

Test a prompt across **multiple questions** to measure consistency:

In [ ]:
# ── Batch evaluation ──
test_questions = [
    "What is an API?",
    "Explain Git branching.",
    "What is the difference between SQL and NoSQL?",
]

system = "You are a senior developer explaining to a junior. Use an analogy. Max 3 sentences."
criteria = "Accuracy, clarity, appropriate analogy, conciseness"

scores = []
print(f"{'Question':<45} {'Score':>6} {'Words':>6}")
print("─" * 60)

for q in test_questions:
    response = ask(q, system=system)
    evaluation = evaluate_response(q, response, criteria)
    score = evaluation.get('score', 0)
    scores.append(score)
    words = len(response.split())
    print(f"{q[:42]+'...':<45} {score:>5}/10 {words:>5}")

avg = sum(scores) / len(scores) if scores else 0
print(f"\n📊 Average score: {avg:.1f}/10")
print(f"   Score range: {min(scores)}-{max(scores)}/10")
print(f"   Consistency: {'🟢 Good' if max(scores)-min(scores) <= 2 else '🟡 Variable'}")

Question                                       Score  Words
────────────────────────────────────────────────────────────
What is an API?...                                9/10    61
Explain Git branching....                         8/10    68
What is the difference between SQL and NoS...     8/10    88

📊 Average score: 8.3/10
   Score range: 8-9/10
   Consistency: 🟢 Good


---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **LLM-as-Judge** | Use an LLM to score other LLM outputs |
| **A/B testing** | Compare prompt variants on same questions |
| **Batch evaluation** | Test across multiple inputs for consistency |
| **Scoring rubric** | Define clear criteria (accuracy, clarity, etc.) |
| **JSON output** | Have the judge return structured scores |

---
**Next:** `06_output_parsing.ipynb` — Parse and structure LLM outputs reliably